# Zuo Sample Refit: Selected Model and Full-profile Asymmetry

This copy fits the previously selected best-BIC CIV model once per Zuo target. The existing fit configuration, broad-only asymmetry, and other profile measurements are retained. It adds `asym_full_profile` and `asym_full_profile_err_mcmc` from the combined broad and optional narrow CIV components, using the same asymmetry measurement convention as the original notebook. New outputs are saved under `asym_full/` folders.

The three literature asymmetries also have separate `_full_profile` values, approximate errors, 0–2 normalizations, and their full-profile wavelength inputs. Existing `CIV_asym_*` columns remain broad-only.


In [1]:
%matplotlib inline
from pathlib import Path
import importlib.util
import tempfile
import timeit
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table

import sys
PROJECT_ROOT = Path('/Users/nayera/PyQSOFit')
sys.path.insert(0, str(PROJECT_ROOT))
from pyqsofit.PyQSOFit import QSOFit
from data_calculations import blueshift_calc, asym_calc
from full_profile_asymmetry import full_profile_results, FULL_PROFILE_COLUMNS, INPUT_COLUMNS, LITERATURE_COLUMNS

warnings.filterwarnings('ignore')
QSOFit.set_mpl_style()


In [ ]:
# Input/output configuration
TARGET_CSV = PROJECT_ROOT / 'Target_lists' / 'zuo_targets_data.csv'
SPECTRA_ROOT = PROJECT_ROOT / 'CIV_galaxy_spectra_targets' / 'zuo_civ_data' / 'base_specs'
FIT_RESULTS_DIR = SPECTRA_ROOT.parent / 'refit_asymmetry' / 'asym_full'
TRACKING_CSV = PROJECT_ROOT / 'Target_lists' / 'asym_full' / 'zuo_civ_batch_fit_inputs_refit_asymmetry.csv'
RESULTS_CSV = PROJECT_ROOT / 'Target_lists' / 'asym_full' / 'zuo_civ_selected_model_outputs_refit_asymmetry.csv'
BEST_RESULTS_CSV = PROJECT_ROOT / 'Target_lists' / 'asym_full' / 'zuo_civ_best_bic_outputs_refit_asymmetry.csv'
SELECTED_MODEL_CSV = PROJECT_ROOT / 'Target_lists' / 'zuo_civ_best_bic_outputs_refit_asymmetry.csv'
PREVIOUS_BEST_RESULTS_CSV = PROJECT_ROOT / 'Target_lists' / 'zuo_civ_best_bic_outputs_refit.csv'
HBETA_CSV = PROJECT_ROOT / 'Target_lists' / 'zuo_civ_hbeta_fwhm_ratio_comparison.csv'
FIGURE_READY_CSV = PROJECT_ROOT / 'Target_lists' / 'asym_full' / 'zuo_civ_figure7_asymmetry_ready.csv'
ASYM_CALC_SCRIPT = Path('/Users/nayera/Astro-research/scripts/asym_calc.py')
WAVE_RANGE = np.array([1445.0, 1705.0])

# MCMC is run once per target's selected model fit.
RUN_MCMC = True
NSAMP = 200
NBURN = 20
NTHIN = 10

# Match the original notebook's low-S/N handling and CIV width limits.
SNR_THRESHOLD = 3.0
LOW_SNR_ERROR_SCALE = 1e6
MAX_CIV_FWHM_KMS_SIGMA = 0.02157
NARROW_FWHM_LIMIT_KMS = 1600
NARROW_CIV_MAXSIG = 0.0022664260627611094
BROAD_FWHM_THRESHOLD = NARROW_FWHM_LIMIT_KMS
CIV_BROAD_PROFILE_GRID_POINTS = 5000
CIV_BROAD_PROFILE_NSIGMA = 6.0

# Load the authoritative literature asymmetry definitions without modifying that script.
asym_spec = importlib.util.spec_from_file_location('authoritative_asym_calc', ASYM_CALC_SCRIPT)
literature_asymmetry = importlib.util.module_from_spec(asym_spec)
asym_spec.loader.exec_module(literature_asymmetry)

MODEL_GRID = {
    'A': {'br_component': 3, 'na_component': 0},
    'B': {'br_component': 2, 'na_component': 1},
    'C': {'br_component': 2, 'na_component': 0},
    'D': {'br_component': 1, 'na_component': 1},
}

# Converted from the SDSS/FITS-redshift rest frame into the Zuo-paper
# rest frame: lambda_zuo = lambda_fits * (1 + z_fits) / (1 + z_zuo).
REST_FRAME_MASKS = {
    '0401': [(1535.361, 1538), (1539,1540.5), (1544.299, 1545.789), (1547.278, 1549.265)],
    '0725': [(1580.850, 1583.841), (1584.837, 1588.824)],
    '0731': [(1539.771, 1542.776), (1547.785, 1550.791)],
    '2608': [(1536.913, 1542.936)],
    '3683': [(1533, 1538)],
    '2825': [(1539.5, 1541.5)],
    '3699': [(1526.811, 1531.822)],
    '3804': [(1504.639, 1508.591), (1526,1527.5), (1546.1,1547.5), (1548.603, 1552.555)],
    '4206': [(1559.599, 1562.098), (1563.597, 1565.596), (1566.595, 1570.093)],
    '4604': [(1510.138, 1512.140), (1534.172, 1536.675), (1538, 1541.182), (1590.251, 1593.255)],
    '5496': [(1529.493, 1531.490)],
}

# Set to a small integer for a smoke test, then back to None for the full run.
TARGET_LIMIT = None


In [3]:
def as_serializable(value):
    if value is None:
        return ''
    arr = np.asarray(value)
    return arr.tolist()


def spectrum_prefix(file_name):
    parts = Path(file_name).stem.split('-')
    if len(parts) >= 2 and parts[0] == 'spec':
        return parts[1]
    return parts[0]


def mask_for_spectrum(file_name):
    regions = REST_FRAME_MASKS.get(spectrum_prefix(file_name))
    return None if regions is None else np.asarray(regions, dtype=float)


REQUIRED_COLUMNS = {'name_sdss', 'z', 'ra', 'dec', 'plate', 'mjd', 'fiber', 'fits_path'}
SOURCE_COLUMNS = None


def inspect_spectrum(fits_path, z):
    if not fits_path.exists():
        return False, np.nan, np.nan, 'missing FITS file'
    try:
        with fits.open(fits_path) as hdul:
            names = set(hdul[1].columns.names)
            missing_arrays = sorted({'loglam', 'flux', 'ivar'} - names)
            if missing_arrays:
                return False, np.nan, np.nan, f'missing FITS arrays: {missing_arrays}'
            obs_wave = 10 ** np.asarray(hdul[1].data['loglam'], dtype=float)
        rest_min = float(np.nanmin(obs_wave) / (1 + z))
        rest_max = float(np.nanmax(obs_wave) / (1 + z))
        if rest_min > WAVE_RANGE[0] or rest_max < WAVE_RANGE[1]:
            reason = f'insufficient rest-frame coverage: {rest_min:.1f}--{rest_max:.1f} A'
            return False, rest_min, rest_max, reason
        return True, rest_min, rest_max, ''
    except Exception as exc:
        return False, np.nan, np.nan, f'cannot read FITS file: {exc!r}'


def load_targets(target_csv=TARGET_CSV):
    global SOURCE_COLUMNS
    df = pd.read_csv(target_csv)
    SOURCE_COLUMNS = list(df.columns)
    missing_columns = sorted(REQUIRED_COLUMNS - set(df.columns))
    if missing_columns:
        raise ValueError(f'Missing required columns in {target_csv.name}: {missing_columns}')

    df = df[df['name_sdss'].notna() & df['fits_path'].notna()].copy()
    df['target_name'] = df['name_sdss'].astype(str).str.strip()
    if df['target_name'].duplicated().any():
        duplicates = df.loc[df['target_name'].duplicated(False), 'target_name'].tolist()
        raise ValueError(f'Duplicate target names in {target_csv.name}: {duplicates}')
    df['file_name'] = df['fits_path'].apply(lambda value: Path(str(value)).name)
    # Resolve against the project folder so the notebook does not depend on CSV absolute paths.
    df['input_fits_path'] = df['fits_path'].astype(str)
    df['fits_path'] = df['file_name'].apply(lambda value: SPECTRA_ROOT / value)
    df['fit_z'] = pd.to_numeric(df['z'], errors='coerce')
    for column in ['ra', 'dec', 'plate', 'mjd', 'fiber']:
        df[column] = pd.to_numeric(df[column], errors='coerce')
    df['fit_z_source'] = 'Zuo2020 z'
    df['range_arr'] = [WAVE_RANGE.copy() for _ in range(len(df))]
    df['mask_arr'] = df['file_name'].apply(mask_for_spectrum)

    inspections = []
    for row in df.itertuples(index=False):
        reasons = []
        if not np.isfinite(row.fit_z) or row.fit_z <= 0:
            valid, rest_min, rest_max = False, np.nan, np.nan
            reasons.append('missing/non-positive fit redshift')
        else:
            valid, rest_min, rest_max, spectrum_reason = inspect_spectrum(
                Path(row.fits_path), float(row.fit_z)
            )
            if spectrum_reason:
                reasons.append(spectrum_reason)
        missing_metadata = [
            column for column in ['ra', 'dec', 'plate', 'mjd', 'fiber']
            if not np.isfinite(getattr(row, column))
        ]
        if missing_metadata:
            valid = False
            reasons.append(f'missing/non-numeric metadata: {missing_metadata}')
        inspections.append((valid, rest_min, rest_max, '; '.join(reasons)))
    df[['spectrum_valid', 'rest_wave_min', 'rest_wave_max', 'skip_reason']] = pd.DataFrame(
        inspections, index=df.index
    )
    df['run_fit'] = df['spectrum_valid'] & df['fit_z'].notna() & (df['fit_z'] > 0)

    export = df.copy()
    export['fits_path'] = export['fits_path'].astype(str)
    export['range_arr'] = export['range_arr'].apply(as_serializable)
    export['mask_arr'] = export['mask_arr'].apply(as_serializable)
    export.to_csv(TRACKING_CSV, index=False)
    return df


TRACKING_CSV.parent.mkdir(parents=True, exist_ok=True)
targets = load_targets()
fit_targets = targets[targets['run_fit']].copy()
if TARGET_LIMIT is not None:
    fit_targets = fit_targets.head(TARGET_LIMIT).copy()

best_models = pd.read_csv(SELECTED_MODEL_CSV, usecols=['Target', 'Model'])
fit_targets['Model'] = fit_targets['target_name'].map(best_models.set_index('Target')['Model'])

print(f'Loaded {len(targets)} Zuo et al. targets from {TARGET_CSV.name}.')
print(f'{len(fit_targets)} targets passed the FITS/redshift/coverage preflight.')
print(f'Tracking table written to {TRACKING_CSV}.')
if (~targets['run_fit']).any():
    display(targets.loc[~targets['run_fit'], ['target_name', 'file_name', 'fit_z', 'skip_reason']])
    raise RuntimeError('Preflight failed; fix the listed inputs before starting the batch fits.')

display(fit_targets[['target_name', 'file_name', 'fit_z', 'range_arr', 'mask_arr', 'rest_wave_min', 'rest_wave_max', 'fits_path']])


Loaded 19 Zuo et al. targets from zuo_targets_data.csv.
19 targets passed the FITS/redshift/coverage preflight.
Tracking table written to /Users/nayera/PyQSOFit/Target_lists/asym_full/zuo_civ_batch_fit_inputs_refit_asymmetry.csv.


,target_name,file_name,fit_z,range_arr,mask_arr,rest_wave_min,rest_wave_max,fits_path
0,J011521.20+152453.3,spec-2825-54439-0077.fits,3.443,"[1445.0, 1705.0]","[[1539.449, 1541.94]]",857.282075,2069.301200,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
1,J014214.75+002324.2,spec-0401-51788-0500.fits,3.379,"[1445.0, 1705.0]","[[1535.361, 1538.34], [1544.299, 1545.789], [1...",865.416128,2099.544470,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
2,J015741.57-010629.6,spec-4233-55449-0178.fits,3.572,"[1445.0, 1705.0]",None,779.818837,2257.843950,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
3,J025021.76-075749.9,spec-0457-51901-0261.fits,3.337,"[1445.0, 1705.0]",None,879.043693,2118.900520,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
4,J025905.63+001121.9,spec-4242-55476-0977.fits,3.373,"[1445.0, 1705.0]",None,821.713748,2374.218715,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
5,J030341.04-002321.9,spec-0709-52205-0106.fits,3.233,"[1445.0, 1705.0]",None,897.949179,2173.460166,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
6,J030449.85-000813.4,spec-0411-51817-0153.fits,3.287,"[1445.0, 1705.0]",None,887.659707,2144.601174,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
7,J075303.34+423130.8,spec-3683-55178-0484.fits,3.590,"[1445.0, 1705.0]",None,775.867024,2248.989660,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
8,J080430.56+542041.1,spec-1871-53384-0234.fits,3.759,"[1445.0, 1705.0]",None,797.782229,1929.230941,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
9,J080819.69+373047.3,spec-3804-55267-0640.fits,3.480,"[1445.0, 1705.0]","[[1504.639, 1508.591], [1548.603, 1552.555]]",799.322222,2310.584483,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...


In [4]:
def continuum_windows_from_range(range_arr):
    lo, hi = map(float, range_arr)
    low_start = max(lo, 1445.0)
    low_end = min(low_start + 20.0, hi)
    high_end = hi
    high_start = max(hi - 15.0, low_end + 5.0)
    if low_end - low_start < 5.0 or high_end - high_start < 5.0 or high_start <= low_end:
        width = hi - lo
        low_start, low_end = lo, lo + 0.15 * width
        high_start, high_end = hi - 0.15 * width, hi
    return np.rec.array(
        [(low_start, low_end), (high_start, high_end)],
        formats='float32,float32',
        names='min,max',
    )


def make_qsopar(param_dir, range_arr, br_component, na_component):
    primary_hdu = fits.PrimaryHDU(header=fits.Header({'Author': 'Nayera Abdessalam'}))

    line_priors = np.rec.array([
        (1549.06, 'CIV', 1450, 1700, 'CIV_br', br_component, 0.0, 0.0, 1e10,
         0.005, 0.004, MAX_CIV_FWHM_KMS_SIGMA, 0.015, 0, 0, 0, 0.05, 1),
        (1549.06, 'CIV', 1450, 1700, 'CIV_na', na_component, 0.0, 0.0, 1e10,
         1e-3, 5e-4, NARROW_CIV_MAXSIG, 0.01, 1, 1, 0, 0.002, 1),
    ], formats='float32,a20,float32,float32,a20,int32,float32,float32,float32,float32,float32,float32,float32,int32,int32,int32,float32,int32',
       names='lambda,compname,minwav,maxwav,linename,ngauss,inisca,minsca,maxsca,inisig,minsig,maxsig,voff,vindex,windex,findex,fvalue,vary')
    hdu1 = fits.BinTableHDU(data=line_priors, name='line_priors')

    hdu2 = fits.BinTableHDU(data=continuum_windows_from_range(range_arr), name='conti_windows')

    conti_priors = np.rec.array([
        ('Fe_uv_norm', 0.0, 0.0, 1e10, 1),
        ('Fe_uv_FWHM', 3000, 1200, 18000, 1),
        ('Fe_uv_shift', 0.0, -0.01, 0.01, 1),
        ('Fe_op_norm', 0.0, 0.0, 1e10, 1),
        ('Fe_op_FWHM', 3000, 1200, 18000, 1),
        ('Fe_op_shift', 0.0, -0.01, 0.01, 1),
        ('PL_norm', 1.0, 0.0, 1e10, 1),
        ('PL_slope', -1.5, -5.0, 3.0, 1),
        ('Blamer_norm', 0.0, 0.0, 1e10, 1),
        ('Balmer_Te', 15000, 10000, 50000, 1),
        ('Balmer_Tau', 0.5, 0.1, 2.0, 1),
        ('conti_a_0', 0.0, None, None, 1),
        ('conti_a_1', 0.0, None, None, 1),
        ('conti_a_2', 0.0, None, None, 1),
    ], formats='a20,float32,float32,float32,int32', names='parname,initial,min,max,vary')
    hdu3 = fits.BinTableHDU(data=conti_priors, name='conti_priors')

    measure_info = Table([[[1350, 1450, 3000, 4200, 5100]], [[[1500, 1550]]]],
                         names=['cont_loc', 'Fe_flux_range'], dtype=['float32', 'float32'])
    hdu4 = fits.BinTableHDU(data=measure_info, name='measure_info')

    param_path = Path(param_dir) / 'qsopar.fits'
    fits.HDUList([primary_hdu, hdu1, hdu2, hdu3, hdu4]).writeto(param_path, overwrite=True)
    return param_path


In [5]:
def read_sdss_spectrum(fits_path):
    with fits.open(fits_path) as data:
        lam = 10 ** data[1].data['loglam']
        flux = np.asarray(data[1].data['flux'], dtype=float)
        ivar = np.asarray(data[1].data['ivar'], dtype=float)
        err = np.full_like(flux, np.inf, dtype=float)
        good_ivar = np.isfinite(ivar) & (ivar > 0)
        err[good_ivar] = 1 / np.sqrt(ivar[good_ivar])

        header = data[0].header
        meta = {
            'ra': header.get('plug_ra', np.nan),
            'dec': header.get('plug_dec', np.nan),
            'plateid': header.get('plateid', -1),
            'mjd': header.get('mjd', -1),
            'fiberid': header.get('fiberid', -1),
        }

    good = np.isfinite(flux) & np.isfinite(err) & (err > 0)
    snr = np.zeros_like(flux, dtype=float)
    snr[good] = np.abs(flux[good]) / err[good]
    low_snr = snr < SNR_THRESHOLD
    err[low_snr] *= LOW_SNR_ERROR_SCALE
    meta['snr_masked_pixels'] = int(np.sum(low_snr))
    meta['total_pixels'] = int(len(flux))
    meta['snr_masked_fraction'] = float(np.sum(low_snr) / len(flux))
    return lam, flux, err, meta


def named_pyqsofit_results(q):
    """Return every scalar continuum and line result with collision-safe names."""
    output = {}
    for prefix, names, values in (
        ('pyqsofit_conti', q.conti_result_name, q.conti_result),
        ('pyqsofit_line', q.line_result_name, q.line_result),
    ):
        for name, value in zip(names, values):
            scalar = np.ravel(value)[0]
            output[f'{prefix}_{name}'] = scalar.item() if hasattr(scalar, 'item') else scalar
    return output


def source_metadata(target_row):
    """Carry the complete Zuo input row into each result for combined analysis."""
    return {
        f'zuo_{column}': (target_row['input_fits_path'] if column == 'fits_path' else target_row[column])
        for column in SOURCE_COLUMNS
    }


def get_line_value(q, key, default=np.nan):
    idx = np.where(np.asarray(q.line_result_name) == key)[0]
    if idx.size == 0:
        return default
    return float(np.ravel(q.line_result[idx[0]])[0])


def component_props(q, line_name, line_type, ln_sigma_br=NARROW_CIV_MAXSIG):
    names = ['FWHM', 'sigma', 'EW', 'peak', 'area', 'snr']
    values = np.array(q.line_prop_from_name(line_name, line_type, ln_sigma_br=ln_sigma_br), dtype=float)
    out = dict(zip(names, values))

    if getattr(q, 'MCMC', False) and len(getattr(q, 'gauss_result_all', [])) > 0:
        samples = []
        for i in range(q.gauss_result_all.shape[0]):
            samples.append(q.line_prop_from_name(line_name, line_type, sample_index=i, ln_sigma_br=ln_sigma_br))
        samples = np.asarray(samples, dtype=float)
        for j, name in enumerate(names):
            finite = samples[:, j][np.isfinite(samples[:, j])]
            if finite.size:
                lo, hi = np.nanpercentile(finite, [16, 84])
                out[f'{name}_err_mcmc'] = float((hi - lo) / 2)
            else:
                out[f'{name}_err_mcmc'] = np.nan
    else:
        for name in names:
            out[f'{name}_err_mcmc'] = np.nan
    return out


def gaussian_params(q, line_name, sample_index=None):
    result_names = np.asarray(q.gauss_result_name)
    result = np.asarray(q.gauss_result, dtype=object)
    params = []
    i = 1
    while True:
        keys = [f'{line_name}_{i}_scale', f'{line_name}_{i}_centerwave', f'{line_name}_{i}_sigma']
        if not all(key in result_names for key in keys):
            break
        if sample_index is None:
            vals = [float(np.ravel(result[result_names == key][0])[0]) for key in keys]
        else:
            # gauss_result_name interleaves value/error, while gauss_result_all stores values only.
            indices = [int(np.flatnonzero(result_names == key)[0] // 2) for key in keys]
            vals = [float(q.gauss_result_all[sample_index, index]) for index in indices]
        params.append(vals)
        i += 1
    return np.asarray(params, dtype=float)


def broad_half_wavelengths(q, sample_index=None):
    params = gaussian_params(q, 'CIV_br', sample_index=sample_index)
    if params.size == 0:
        return np.nan, np.nan, np.nan
    lam = np.asarray(q.wave, dtype=float)
    sel = (lam >= 1450.0) & (lam <= 1700.0)
    if np.sum(sel) < 3:
        sel = np.ones_like(lam, dtype=bool)
    lam_w = lam[sel]
    lnlam = np.log(lam_w)
    model = np.zeros_like(lam_w, dtype=float)
    for amp, center, sigma in params:
        model += amp * np.exp(-0.5 * ((lnlam - center) / sigma) ** 2)
    if not np.any(np.isfinite(model)) or np.nanmax(model) <= 0:
        return np.nan, np.nan, np.nan

    peak = np.nanmax(model)
    i_peak = int(np.nanargmax(model))
    half_peak = 0.5 * peak

    def crossing_left():
        below = np.where(model[:i_peak] < half_peak)[0]
        if below.size == 0:
            return float(lam_w[0])
        j0 = below[-1]
        j1 = j0 + 1
        return float(np.interp(half_peak, [model[j0], model[j1]], [lam_w[j0], lam_w[j1]]))

    def crossing_right():
        below = np.where(model[i_peak + 1:] < half_peak)[0]
        if below.size == 0:
            return float(lam_w[-1])
        j1 = i_peak + 1 + below[0]
        j0 = j1 - 1
        return float(np.interp(half_peak, [model[j0], model[j1]], [lam_w[j0], lam_w[j1]]))

    lam_blue = crossing_left()
    lam_red = crossing_right()

    increments = 0.5 * (model[1:] + model[:-1]) * np.diff(lam_w)
    cum = np.cumsum(increments)
    if cum.size == 0 or cum[-1] <= 0:
        lam_half = np.nan
    else:
        lam_half = float(np.interp(0.5 * cum[-1], cum, lam_w[1:]))
    return lam_blue, lam_red, lam_half


BROAD_ASYMMETRY_INPUT_KEYS = (
    'lam_p05', 'lam_p50', 'lam_p95',
    'lam_blue_15pct', 'lam_red_15pct',
    'lam_blue_25pct', 'lam_red_25pct',
    'lam_blue_80pct', 'lam_red_80pct',
    'lam_peak',
)


def broad_asymmetry_input_wavelengths(q, sample_index=None):
    """Measure shape wavelengths from the continuous broad-C IV-only model."""
    empty = {key: np.nan for key in BROAD_ASYMMETRY_INPUT_KEYS}
    params = gaussian_params(q, 'CIV_br', sample_index=sample_index)
    if params.size == 0:
        return empty

    lnlam = np.linspace(
        np.min(params[:, 1] - CIV_BROAD_PROFILE_NSIGMA * params[:, 2]),
        np.max(params[:, 1] + CIV_BROAD_PROFILE_NSIGMA * params[:, 2]),
        CIV_BROAD_PROFILE_GRID_POINTS,
    )
    lam = np.exp(lnlam)
    model = np.zeros_like(lam, dtype=float)
    for amp, center, sigma in params:
        model += amp * np.exp(-0.5 * ((lnlam - center) / sigma) ** 2)
    if not np.any(np.isfinite(model)) or np.nanmax(model) <= 0:
        return empty

    i_peak = int(np.nanargmax(model))
    peak_flux = float(model[i_peak])

    def interpolate_crossing(j0, j1, target):
        y0, y1 = model[j0], model[j1]
        if y1 == y0:
            return np.nan
        return float(lam[j0] + (target - y0) * (lam[j1] - lam[j0]) / (y1 - y0))

    def crossings(fraction):
        target = fraction * peak_flux
        left_below = np.where(model[:i_peak] < target)[0]
        right_below = np.where(model[i_peak + 1:] < target)[0]
        if left_below.size == 0 or right_below.size == 0:
            return np.nan, np.nan
        left0 = int(left_below[-1])
        right1 = int(i_peak + 1 + right_below[0])
        return (
            interpolate_crossing(left0, left0 + 1, target),
            interpolate_crossing(right1 - 1, right1, target),
        )

    lam_blue_15pct, lam_red_15pct = crossings(0.15)
    lam_blue_25pct, lam_red_25pct = crossings(0.25)
    lam_blue_80pct, lam_red_80pct = crossings(0.80)

    increments = 0.5 * (model[1:] + model[:-1]) * np.diff(lam)
    cumulative = np.concatenate(([0.0], np.cumsum(increments)))
    if cumulative[-1] <= 0:
        return empty
    lam_p05, lam_p50, lam_p95 = np.interp(
        np.array([0.05, 0.50, 0.95]) * cumulative[-1], cumulative, lam
    )

    return {
        'lam_p05': float(lam_p05),
        'lam_p50': float(lam_p50),
        'lam_p95': float(lam_p95),
        'lam_blue_15pct': lam_blue_15pct,
        'lam_red_15pct': lam_red_15pct,
        'lam_blue_25pct': lam_blue_25pct,
        'lam_red_25pct': lam_red_25pct,
        'lam_blue_80pct': lam_blue_80pct,
        'lam_red_80pct': lam_red_80pct,
        'lam_peak': float(lam[i_peak]),
    }


def full_profile_asymmetry(q, sample_index=None):
    """Apply the existing asymmetry method to broad plus narrow CIV Gaussians."""
    broad_params = gaussian_params(q, 'CIV_br', sample_index=sample_index).reshape(-1, 3)
    narrow_params = gaussian_params(q, 'CIV_na', sample_index=sample_index).reshape(-1, 3)
    params = np.concatenate([broad_params, narrow_params], axis=0)
    if params.size == 0:
        return np.nan
    lam = np.asarray(q.wave, dtype=float)
    sel = (lam >= 1450.0) & (lam <= 1700.0)
    if np.sum(sel) < 3:
        sel = np.ones_like(lam, dtype=bool)
    lam_w = lam[sel]
    lnlam = np.log(lam_w)
    model = np.zeros_like(lam_w, dtype=float)
    for amp, center, sigma in params:
        model += amp * np.exp(-0.5 * ((lnlam - center) / sigma) ** 2)
    if not np.any(np.isfinite(model)) or np.nanmax(model) <= 0:
        return np.nan

    peak = np.nanmax(model)
    i_peak = int(np.nanargmax(model))
    half_peak = 0.5 * peak

    def crossing_left():
        below = np.where(model[:i_peak] < half_peak)[0]
        if below.size == 0:
            return float(lam_w[0])
        j0 = below[-1]
        j1 = j0 + 1
        return float(np.interp(half_peak, [model[j0], model[j1]], [lam_w[j0], lam_w[j1]]))

    def crossing_right():
        below = np.where(model[i_peak + 1:] < half_peak)[0]
        if below.size == 0:
            return float(lam_w[-1])
        j1 = i_peak + 1 + below[0]
        j0 = j1 - 1
        return float(np.interp(half_peak, [model[j0], model[j1]], [lam_w[j0], lam_w[j1]]))

    lam_blue = crossing_left()
    lam_red = crossing_right()
    if narrow_params.size == 0:
        # This is the same peak used by the existing broad-only asymmetry.
        peak_wavelength = q.line_prop_from_name(
            'CIV_br', 'broad',
            sample_index=-1 if sample_index is None else sample_index,
            ln_sigma_br=NARROW_CIV_MAXSIG,
        )[3]
    else:
        # PyQSOFit's line_prop peak grid, applied to the combined components.
        left = np.min(params[:, 1] - 3 * params[:, 2])
        right = np.max(params[:, 1] + 3 * params[:, 2])
        disp = 1e-4 * np.log(10)
        npix = int((right - left) / disp)
        xx = np.linspace(left, right, npix)
        model_peak = np.zeros_like(xx)
        for amp, center, sigma in params:
            model_peak += amp * np.exp(-0.5 * ((xx - center) / sigma) ** 2)
        peak_wavelength = np.exp(xx[np.argmax(model_peak)])
    return float(asym_calc(lam_blue, lam_red, peak_wavelength))


def posterior_error(values):
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return np.nan
    lo, hi = np.nanpercentile(finite, [16, 84])
    return float((hi - lo) / 2)


def broad_profile_mcmc_results(q):
    samples = getattr(q, 'gauss_result_all', None)
    if not getattr(q, 'MCMC', False) or samples is None or len(samples) == 0:
        return {
            'CIV_blueshift_err_mcmc': np.nan,
            'CIV_asym_err_mcmc': np.nan,
            'lam_blue_err_mcmc': np.nan,
            'lam_red_err_mcmc': np.nan,
            'lam_half_err_mcmc': np.nan,
            **{f'{key}_err_mcmc': np.nan for key in BROAD_ASYMMETRY_INPUT_KEYS},
            'mcmc_samples_returned': 0,
            'mcmc_valid_blueshift_samples': 0,
            'mcmc_valid_asym_samples': 0,
        }

    n_samples = samples.shape[0]
    profiles = [broad_half_wavelengths(q, sample_index=i) for i in range(n_samples)]
    lam_blue = np.asarray([profile[0] for profile in profiles], dtype=float)
    lam_red = np.asarray([profile[1] for profile in profiles], dtype=float)
    lam_half = np.asarray([profile[2] for profile in profiles], dtype=float)
    asymmetry_inputs = [
        broad_asymmetry_input_wavelengths(q, sample_index=i) for i in range(n_samples)
    ]
    asymmetry_input_samples = {
        key: np.asarray([values[key] for values in asymmetry_inputs], dtype=float)
        for key in BROAD_ASYMMETRY_INPUT_KEYS
    }

    # Use PyQSOFit's continuous-model broad-line peak for each posterior sample.
    broad_properties = np.asarray([
        q.line_prop_from_name(
            'CIV_br', 'broad', sample_index=i, ln_sigma_br=NARROW_CIV_MAXSIG
        )
        for i in range(n_samples)
    ], dtype=float)
    br_peak = broad_properties[:, 3]

    blueshift = np.asarray([blueshift_calc(value) for value in lam_half], dtype=float)
    # asym_calc is imported from data_calculations.py, the authoritative equation.
    asymmetry = np.asarray([
        asym_calc(blue, red, peak)
        for blue, red, peak in zip(lam_blue, lam_red, br_peak)
    ], dtype=float)

    return {
        'CIV_blueshift_err_mcmc': posterior_error(blueshift),
        'CIV_asym_err_mcmc': posterior_error(asymmetry),
        'lam_blue_err_mcmc': posterior_error(lam_blue),
        'lam_red_err_mcmc': posterior_error(lam_red),
        'lam_half_err_mcmc': posterior_error(lam_half),
        **{
            f'{key}_err_mcmc': posterior_error(values)
            for key, values in asymmetry_input_samples.items()
        },
        'mcmc_samples_returned': int(n_samples),
        'mcmc_valid_blueshift_samples': int(np.isfinite(blueshift).sum()),
        'mcmc_valid_asym_samples': int(np.isfinite(asymmetry).sum()),
    }


In [6]:
def fit_target_model(target_row, model_name, model_cfg):
    target = target_row['target_name']
    fits_path = Path(target_row['fits_path'])
    z = float(target_row['fit_z'])
    range_arr = np.asarray(target_row['range_arr'], dtype=float)
    mask_arr = target_row['mask_arr']
    if mask_arr is not None:
        mask_arr = np.asarray(mask_arr, dtype=float)
    output_stem = f'{target}_{model_name}'
    output_fits_path = FIT_RESULTS_DIR / f'{output_stem}.fits'
    output_pdf_path = FIT_RESULTS_DIR / f'{output_stem}.pdf'

    lam, flux, err, meta = read_sdss_spectrum(fits_path)

    with tempfile.TemporaryDirectory(prefix='pyqsofit_civ_') as param_dir:
        make_qsopar(param_dir, range_arr, model_cfg['br_component'], model_cfg['na_component'])
        q = QSOFit(lam, flux, err, z,
                   ra=meta['ra'], dec=meta['dec'], plateid=meta['plateid'],
                   mjd=meta['mjd'], fiberid=meta['fiberid'], path=param_dir)

        start = timeit.default_timer()
        q.Fit(name=output_stem,
              nsmooth=1,
              and_mask=False,
              or_mask=False,
              reject_badpix=True,
              deredden=False,
              wave_range=range_arr,
              wave_mask=mask_arr,
              decompose_host=False,
              host_prior=False,
              host_prior_scale=0.2,
              host_line_mask=True,
              decomp_na_mask=True,
              qso_type='CZBIN1',
              npca_qso=10,
              host_type='PCA',
              npca_gal=5,
              Fe_uv_op=False,
              poly=False,
              BC=False,
              initial_guess=None,
              rej_abs_conti=True,
              n_pix_min_conti=100,
              linefit=True,
              rej_abs_line=True,
              MC=False,
              MCMC=RUN_MCMC,
              nsamp=NSAMP,
              param_file_name='qsopar.fits',
              nburn=NBURN,
              nthin=NTHIN,
              epsilon_jitter=0.,
              save_result=False,
              save_fits_path=str(FIT_RESULTS_DIR),
              save_fits_name=output_stem,
              plot_fig=False,
              save_fig=False,
              plot_corner=False,
              verbose=False,
              kwargs_plot={'broad_fwhm': BROAD_FWHM_THRESHOLD,
                           'save_fig_path': str(FIT_RESULTS_DIR)},
              kwargs_conti_emcee={},
              kwargs_line_emcee={})
        fit_seconds = float(timeit.default_timer() - start)

    br = component_props(q, 'CIV_br', 'broad', ln_sigma_br=NARROW_CIV_MAXSIG)
    na = component_props(q, 'CIV_na', 'narrow', ln_sigma_br=NARROW_CIV_MAXSIG)
    lam_blue, lam_red, lam_half = broad_half_wavelengths(q)
    blueshift = blueshift_calc(lam_half)
    asymmetry = asym_calc(lam_blue, lam_red, br['peak'])
    asymmetry_inputs = broad_asymmetry_input_wavelengths(q)
    profile_mcmc = broad_profile_mcmc_results(q)
    full_asymmetry = full_profile_results(q, gaussian_params, full_profile_asymmetry, literature_asymmetry)

    row = {
        'Target': target,
        'Model': model_name,
        'file_name': target_row['file_name'],
        'fits_path': str(fits_path),
        'fit_z': z,
        'fit_z_source': target_row['fit_z_source'],
        'range_arr': as_serializable(range_arr),
        'mask_arr': as_serializable(mask_arr),
        'output_fits_path': str(output_fits_path),
        'output_pdf_path': str(output_pdf_path),
        'br_components_forced': model_cfg['br_component'],
        'na_components_forced': model_cfg['na_component'],
        'narrow_fwhm_limit_kms': NARROW_FWHM_LIMIT_KMS,
        'lam_blue': lam_blue,
        'lam_red': lam_red,
        'lam_half': lam_half,
        'br_peak': br['peak'],
        'br_peak_err_mcmc': br['peak_err_mcmc'],
        'CIV_blueshift_kms': blueshift,
        'CIV_blueshift_err_mcmc': profile_mcmc['CIV_blueshift_err_mcmc'],
        'CIV_asym': asymmetry,
        'CIV_asym_err_mcmc': profile_mcmc['CIV_asym_err_mcmc'],
        **full_asymmetry,
        'lam_blue_err_mcmc': profile_mcmc['lam_blue_err_mcmc'],
        'lam_red_err_mcmc': profile_mcmc['lam_red_err_mcmc'],
        'lam_half_err_mcmc': profile_mcmc['lam_half_err_mcmc'],
        'CIV_lam_p05_A': asymmetry_inputs['lam_p05'],
        'CIV_lam_p05_err_mcmc_A': profile_mcmc['lam_p05_err_mcmc'],
        'CIV_lam_p50_A': asymmetry_inputs['lam_p50'],
        'CIV_lam_p50_err_mcmc_A': profile_mcmc['lam_p50_err_mcmc'],
        'CIV_lam_p95_A': asymmetry_inputs['lam_p95'],
        'CIV_lam_p95_err_mcmc_A': profile_mcmc['lam_p95_err_mcmc'],
        'CIV_lam_blue_15pct_A': asymmetry_inputs['lam_blue_15pct'],
        'CIV_lam_blue_15pct_err_mcmc_A': profile_mcmc['lam_blue_15pct_err_mcmc'],
        'CIV_lam_red_15pct_A': asymmetry_inputs['lam_red_15pct'],
        'CIV_lam_red_15pct_err_mcmc_A': profile_mcmc['lam_red_15pct_err_mcmc'],
        'CIV_lam_blue_25pct_A': asymmetry_inputs['lam_blue_25pct'],
        'CIV_lam_blue_25pct_err_mcmc_A': profile_mcmc['lam_blue_25pct_err_mcmc'],
        'CIV_lam_red_25pct_A': asymmetry_inputs['lam_red_25pct'],
        'CIV_lam_red_25pct_err_mcmc_A': profile_mcmc['lam_red_25pct_err_mcmc'],
        'CIV_lam_blue_80pct_A': asymmetry_inputs['lam_blue_80pct'],
        'CIV_lam_blue_80pct_err_mcmc_A': profile_mcmc['lam_blue_80pct_err_mcmc'],
        'CIV_lam_red_80pct_A': asymmetry_inputs['lam_red_80pct'],
        'CIV_lam_red_80pct_err_mcmc_A': profile_mcmc['lam_red_80pct_err_mcmc'],
        'CIV_lam_peak_A': asymmetry_inputs['lam_peak'],
        'CIV_lam_peak_err_mcmc_A': profile_mcmc['lam_peak_err_mcmc'],
        'mcmc_samples_returned': profile_mcmc['mcmc_samples_returned'],
        'mcmc_valid_blueshift_samples': profile_mcmc['mcmc_valid_blueshift_samples'],
        'mcmc_valid_asym_samples': profile_mcmc['mcmc_valid_asym_samples'],
        'reduced Chi squared': get_line_value(q, '1_line_red_chi2'),
        'BIC_mcmc': get_line_value(q, '1_line_bic'),
        'FWHM_CIV': br['FWHM'],
        'FWHM_err_mcmc': br['FWHM_err_mcmc'],
        'br_EW': br['EW'],
        'br_EW_err_mcmc': br['EW_err_mcmc'],
        'br_area': br['area'],
        'br_area_err_mcmc': br['area_err_mcmc'],
        'br_sigma': br['sigma'],
        'br_sigma_err_mcmc': br['sigma_err_mcmc'],
        'br_snr': br['snr'],
        'na_FWHM': na['FWHM'],
        'na_FWHM_err_mcmc': na['FWHM_err_mcmc'],
        'na_EW': na['EW'],
        'na_EW_err_mcmc': na['EW_err_mcmc'],
        'na_area': na['area'],
        'na_area_err_mcmc': na['area_err_mcmc'],
        'na_sigma': na['sigma'],
        'na_sigma_err_mcmc': na['sigma_err_mcmc'],
        'na_peak': na['peak'],
        'na_peak_err_mcmc': na['peak_err_mcmc'],
        'na_snr': na['snr'],
        'line_min_chi2': get_line_value(q, '1_line_min_chi2'),
        'line_status': get_line_value(q, '1_line_status'),
        'line_ndof': get_line_value(q, '1_ndof'),
        'line_niter': get_line_value(q, '1_niter'),
        'snr_masked_pixels': meta['snr_masked_pixels'],
        'total_pixels': meta['total_pixels'],
        'snr_masked_fraction': meta['snr_masked_fraction'],
        'fit_seconds': fit_seconds,
        'status': 'ok',
        'error': '',
    }
    row.update(source_metadata(target_row))
    row.update(named_pyqsofit_results(q))
    return row


In [7]:
FIT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results = []
expected = len(fit_targets)
print(f'Running {expected} fits: one selected model per target.')

for _, target_row in fit_targets.iterrows():
    model_name = target_row['Model']
    model_cfg = MODEL_GRID[model_name]
    label = f"{target_row['target_name']} model {model_name}"
    print(f'Starting {label}')
    try:
        row = fit_target_model(target_row, model_name, model_cfg)
        print(
            f"Finished {label}: BIC={row['BIC_mcmc']:.3g}, "
            f"FWHM={row['FWHM_CIV']:.3g} km/s, "
            f"blueshift_err={row['CIV_blueshift_err_mcmc']:.3g} km/s, "
            f"asym_err={row['CIV_asym_err_mcmc']:.3g}"
        )
    except Exception as exc:
        row = {
            'Target': target_row['target_name'],
            'Model': model_name,
            'file_name': target_row['file_name'],
            'fits_path': str(target_row['fits_path']),
            'fit_z': target_row['fit_z'],
            'fit_z_source': target_row['fit_z_source'],
            'range_arr': as_serializable(target_row['range_arr']),
            'mask_arr': as_serializable(target_row['mask_arr']),
            'output_fits_path': str(FIT_RESULTS_DIR / f"{target_row['target_name']}_{model_name}.fits"),
            'output_pdf_path': str(FIT_RESULTS_DIR / f"{target_row['target_name']}_{model_name}.pdf"),
            'br_components_forced': model_cfg['br_component'],
            'na_components_forced': model_cfg['na_component'],
            'narrow_fwhm_limit_kms': NARROW_FWHM_LIMIT_KMS,
            **{column: np.nan for column in FULL_PROFILE_COLUMNS},
            'status': 'failed',
            'error': repr(exc),
        }
        row.update(source_metadata(target_row))
        print(f'FAILED {label}: {exc!r}')
    results.append(row)
    pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)

results_df = pd.DataFrame(results)
if 'BIC_mcmc' not in results_df:
    results_df['BIC_mcmc'] = np.nan
else:
    results_df['BIC_mcmc'] = pd.to_numeric(results_df['BIC_mcmc'], errors='coerce')
successful = results_df['status'].eq('ok') & np.isfinite(results_df['BIC_mcmc'])
results_df['delta_BIC'] = np.nan
results_df['is_best_BIC'] = False
if successful.any():
    best_bic_by_target = results_df.loc[successful].groupby('Target')['BIC_mcmc'].transform('min')
    results_df.loc[successful, 'delta_BIC'] = (
        results_df.loc[successful, 'BIC_mcmc'] - best_bic_by_target
    )
    best_indices = results_df.loc[successful].groupby('Target', sort=False)['BIC_mcmc'].idxmin()
    results_df.loc[best_indices, 'is_best_BIC'] = True
    best_results_df = results_df.loc[best_indices].copy().reset_index(drop=True)
else:
    best_results_df = results_df.iloc[0:0].copy()

# Calculate native and fixed-scale asymmetries with the authoritative script.
literature_asymmetry.require_columns(best_results_df)
best_results_df = literature_asymmetry.add_asymmetry_columns(best_results_df)

raw_profile_columns = [
    'CIV_lam_p05_A', 'CIV_lam_p05_err_mcmc_A',
    'CIV_lam_p50_A', 'CIV_lam_p50_err_mcmc_A',
    'CIV_lam_p95_A', 'CIV_lam_p95_err_mcmc_A',
    'CIV_lam_blue_15pct_A', 'CIV_lam_blue_15pct_err_mcmc_A',
    'CIV_lam_red_15pct_A', 'CIV_lam_red_15pct_err_mcmc_A',
    'CIV_lam_blue_25pct_A', 'CIV_lam_blue_25pct_err_mcmc_A',
    'CIV_lam_red_25pct_A', 'CIV_lam_red_25pct_err_mcmc_A',
    'CIV_lam_blue_80pct_A', 'CIV_lam_blue_80pct_err_mcmc_A',
    'CIV_lam_red_80pct_A', 'CIV_lam_red_80pct_err_mcmc_A',
    'CIV_lam_peak_A', 'CIV_lam_peak_err_mcmc_A',
]
asymmetry_columns = [
    'CIV_asym', 'CIV_asym_err_mcmc',
    'CIV_asym_zg_R', 'CIV_asym_zg_R_err_approx',
    'CIV_asym_zg_R_norm_0_2', 'CIV_asym_zg_R_norm_0_2_err_approx',
    'CIV_asym_wolf_15_80', 'CIV_asym_wolf_15_80_err_approx',
    'CIV_asym_wolf_15_80_norm_0_2', 'CIV_asym_wolf_15_80_norm_0_2_err_approx',
    'CIV_asym_4de1_AI_1_4', 'CIV_asym_4de1_AI_1_4_err_approx',
    'CIV_asym_4de1_AI_1_4_norm_0_2', 'CIV_asym_4de1_AI_1_4_norm_0_2_err_approx',
]

hbeta = pd.read_csv(HBETA_CSV)
hbeta_merge = hbeta[[
    'name_sdss', 'zuo_hbeta_fwhm_km_s', 'zuo_hbeta_fwhm_err_km_s'
]].rename(columns={
    'name_sdss': 'Target',
    'zuo_hbeta_fwhm_km_s': 'fwhm_hb',
    'zuo_hbeta_fwhm_err_km_s': 'fwhm_hb_err',
})
figure_ready_df = best_results_df.merge(
    hbeta_merge, on='Target', how='left', validate='one_to_one', indicator=True
)
figure_ready_df.insert(2, 'sample', 'Zuo refit')

# Strict validation: fail loudly rather than removing any object.
assert len(fit_targets) == 19 and fit_targets['target_name'].nunique() == 19
assert len(results_df) == 19
assert not results_df.duplicated(['Target', 'Model']).any()
assert len(best_results_df) == 19 and best_results_df['Target'].nunique() == 19
assert best_results_df['Model'].notna().all()
assert best_results_df['status'].eq('ok').all() and np.isfinite(best_results_df['BIC_mcmc']).all()
assert set(best_results_df['Target']) == set(fit_targets['target_name'])  # no blueshift filtering
assert figure_ready_df['_merge'].eq('both').all()
assert len(figure_ready_df) == 19 and figure_ready_df['Target'].nunique() == 19
positive_columns = ['FWHM_CIV', 'FWHM_err_mcmc', 'fwhm_hb', 'fwhm_hb_err']
assert np.isfinite(figure_ready_df[positive_columns]).all().all()
assert (figure_ready_df[positive_columns] > 0).all().all()
new_numeric_columns = raw_profile_columns + asymmetry_columns + list(FULL_PROFILE_COLUMNS)
assert np.isfinite(figure_ready_df[new_numeric_columns]).all().all()
error_columns = [column for column in new_numeric_columns if '_err_' in column]
assert (figure_ready_df[error_columns] >= 0).all().all()
normalized_columns = [
    'CIV_asym_zg_R_norm_0_2',
    'CIV_asym_wolf_15_80_norm_0_2',
    'CIV_asym_4de1_AI_1_4_norm_0_2',
    'CIV_asym_zg_R_full_profile_norm_0_2',
    'CIV_asym_wolf_15_80_full_profile_norm_0_2',
    'CIV_asym_4de1_AI_1_4_full_profile_norm_0_2',
]
assert ((figure_ready_df[normalized_columns] >= 0) & (figure_ready_df[normalized_columns] <= 2)).all().all()

figure_columns = [
    'Target', 'Model', 'sample', 'FWHM_CIV', 'FWHM_err_mcmc',
    'fwhm_hb', 'fwhm_hb_err', *asymmetry_columns, *raw_profile_columns, *FULL_PROFILE_COLUMNS,
    'BIC_mcmc', 'status',
]
figure_ready_df = figure_ready_df[figure_columns].copy()
results_df.to_csv(RESULTS_CSV, index=False)
best_results_df.to_csv(BEST_RESULTS_CSV, index=False)
figure_ready_df.to_csv(FIGURE_READY_CSV, index=False)

print(f'Validated {len(fit_targets)} unique inputs, {len(results_df)} model rows, and {len(best_results_df)} selected rows.')
print(f'Successful finite-BIC fits: {successful.sum()}; failed/non-finite fits: {(~successful).sum()}')
print('Selected models and BIC values:')
display(best_results_df[['Target', 'Model', 'BIC_mcmc']])
print('Finite-value counts for new columns:')
display(figure_ready_df[new_numeric_columns].apply(lambda column: np.isfinite(column).sum()).rename('finite_count'))
if PREVIOUS_BEST_RESULTS_CSV.exists():
    previous = pd.read_csv(PREVIOUS_BEST_RESULTS_CSV)[['Target', 'Model', 'BIC_mcmc', 'CIV_asym', 'FWHM_CIV']]
    comparison = best_results_df[['Target', 'Model', 'BIC_mcmc', 'CIV_asym', 'FWHM_CIV']].merge(
        previous, on='Target', how='left', suffixes=('_new', '_previous'), validate='one_to_one'
    )
    comparison['model_changed'] = comparison['Model_new'] != comparison['Model_previous']
    comparison['CIV_asym_change'] = comparison['CIV_asym_new'] - comparison['CIV_asym_previous']
    comparison['FWHM_CIV_change'] = comparison['FWHM_CIV_new'] - comparison['FWHM_CIV_previous']
    print('Comparison with the previous refit:')
    display(comparison)
print(f'Selected-model output: {RESULTS_CSV}')
print(f'Minimum-BIC output: {BEST_RESULTS_CSV}')
print(f'Figure-ready output ({len(figure_ready_df)} rows): {FIGURE_READY_CSV}')


Running 19 fits: one selected model per target.
Starting J011521.20+152453.3 model A


100%|██████████| 200/200 [00:02<00:00, 80.24it/s] 


The chain is shorter than 50 times the integrated autocorrelation time for 2 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 4;
tau: [11.98088249 11.8623563 ]


100%|██████████| 200/200 [00:02<00:00, 84.35it/s]


The chain is shorter than 50 times the integrated autocorrelation time for 9 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 4;
tau: [18.29655049 18.33973614 18.70553334 18.88760509 17.52100452 18.64369638
 18.75708326 18.40286171 18.37845745]
Finished J011521.20+152453.3 model A: BIC=395, FWHM=6.53e+03 km/s, blueshift_err=44.8 km/s, asym_err=0.0323
Starting J014214.75+002324.2 model A


100%|██████████| 200/200 [00:01<00:00, 161.19it/s]


... (73 outputs left out)

CIV_lam_p05_A                                 19
CIV_lam_p05_err_mcmc_A                        19
CIV_lam_p50_A                                 19
CIV_lam_p50_err_mcmc_A                        19
CIV_lam_p95_A                                 19
                                              ..
CIV_lam_blue_80pct_full_profile_err_mcmc_A    19
CIV_lam_red_80pct_full_profile_A              19
CIV_lam_red_80pct_full_profile_err_mcmc_A     19
CIV_lam_peak_full_profile_A                   19
CIV_lam_peak_full_profile_err_mcmc_A          19
Name: finite_count, Length: 68, dtype: int64

Comparison with the previous refit:


,Target,Model_new,BIC_mcmc_new,CIV_asym_new,FWHM_CIV_new,Model_previous,BIC_mcmc_previous,CIV_asym_previous,FWHM_CIV_previous,model_changed,CIV_asym_change,FWHM_CIV_change
0,J011521.20+152453.3,A,395.471620,0.615807,6534.826348,A,395.471620,0.615807,6534.826348,False,0.000000e+00,0.000000
1,J014214.75+002324.2,A,310.178578,0.998813,4853.937355,A,310.178578,0.996606,4853.850538,False,2.207326e-03,0.086817
2,J015741.57-010629.6,B,393.515023,1.133791,6670.873838,B,393.515023,1.133791,6670.873838,False,0.000000e+00,0.000000
3,J025021.76-075749.9,B,313.209409,1.056987,8406.739690,B,313.209409,1.056987,8406.739690,False,2.220446e-16,0.000000
4,J025905.63+001121.9,D,262.717563,1.005401,9331.899342,D,262.717563,1.005401,9331.899342,False,0.000000e+00,0.000000
5,J030341.04-002321.9,A,342.612755,0.621756,6522.282189,A,342.612737,0.621826,6522.277594,False,-6.973495e-05,0.004594
6,J030449.85-000813.4,B,384.912413,0.995433,4352.493649,B,384.912413,0.995433,4352.493649,False,0.000000e+00,0.000000
7,J075303.34+423130.8,B,403.105398,1.073496,5748.266802,B,403.105398,1.073496,5748.266802,False,0.000000e+00,0.000000
8,J080430.56+542041.1,A,78.387115,0.727795,7035.573532,A,78.387115,0.727795,7035.573532,False,0.000000e+00,0.000000
9,J080819.69+373047.3,A,160.952767,1.057020,6549.992867,A,160.952767,1.057020,6549.992867,False,0.000000e+00,0.000000


Selected-model output: /Users/nayera/PyQSOFit/Target_lists/asym_full/zuo_civ_selected_model_outputs_refit_asymmetry.csv
Minimum-BIC output: /Users/nayera/PyQSOFit/Target_lists/asym_full/zuo_civ_best_bic_outputs_refit_asymmetry.csv
Figure-ready output (19 rows): /Users/nayera/PyQSOFit/Target_lists/asym_full/zuo_civ_figure7_asymmetry_ready.csv


## Full-profile asymmetry and outputs

The existing broad-only asymmetry and shape measurements retain their original definitions. `asym_full_profile` uses the same 1450–1700 Å wavelength grid and half-maximum crossing interpolation as `CIV_asym`, adding the fitted narrow CIV Gaussian when the selected model has one. Its peak follows the original PyQSOFit peak-grid convention for the combined profile. For each MCMC sample, the broad and narrow parameters come from the same sample; `asym_full_profile_err_mcmc` is half the 16th–84th percentile interval.

The selected model for each of the 19 targets comes from `zuo_civ_best_bic_outputs_refit_asymmetry.csv`. CSV outputs are under `Target_lists/asym_full/`, and model files are under `CIV_galaxy_spectra_targets/zuo_civ_data/refit_asymmetry/asym_full/`.
